# 7. Working with Stellar Catalogs

In this tutorial, you will learn how to query the internal database for identified stars and inspect their cross-session observational data.


## 7.1 Querying the Stellar Registry

The `astrometrics.stars` registry stores cross-session data for stars detected across all your targets.


In [1]:
from astrometricslib import Astrometrics

astrometrics = Astrometrics()

# Count the identified stars, then read the first few full records
answer = astrometrics.stars.query(detail="objects", limit=10)
stars = answer.objects
print(f"Total stars tracked in the database: {answer.total_matching}")

if stars:
    # Inspect the first star
    star = stars[0]
    print(f"Star ID: {star.id}")
    print(f"Coordinates: RA {star.right_ascension}, DEC {star.declination}")
    print(f"Number of observing sessions matched: {len(star.session_matches)}")

Total stars tracked in the database: 562
Star ID: 1SWASP J164159.14+363818.5
Coordinates: RA 250.49639746514987, DEC 36.63889456639027
Number of observing sessions matched: 0


## 7.2 Inspecting Light Curves

A `StellarObject` may contain a `PhotometryResult` object if photometry has been run on it.


In [2]:
# Find the first star that actually has light-curve data
star_with_light_curve = next(
    (
        candidate_star
        for candidate_star in stars
        if candidate_star.photometry and candidate_star.photometry.timestamps
    ),
    None,
)

if star_with_light_curve:
    light_curve = star_with_light_curve.photometry
    print(f"Star ID: {star_with_light_curve.id}")
    print(f"Light curve points: {len(light_curve.timestamps)}")
    print(f"First flux value: {light_curve.fluxes[0]:.4f}")
else:
    print("No light curve data found for any star.")

Star ID: 1SWASP J164159.14+363818.5
Light curve points: 5
First flux value: 1023.6633


## 7.3 A Star's Spectrum

A `StellarObject` may also contain a `SpectroscopyResult` object if spectroscopy has been run on it. `astrometrics.stars.get_spectrum(star_id)` reads that object from the star's id alone, so you do not need to hold the star record.

In [3]:
# Find the first star that has a spectrum, then read it from the star
spectrum_ids = astrometrics.stars.query(has_spectra=True, detail="ids", limit=1).ids or []
star_with_spectrum = astrometrics.stars.get(spectrum_ids[0]) if spectrum_ids else None

if star_with_spectrum:
    spectroscopy_result = astrometrics.stars.get_spectrum(
        star_with_spectrum.id
    )  # a SpectroscopyResult object
    print(f"Star ID: {star_with_spectrum.id}")
    print(f"Wavelength points: {len(spectroscopy_result.wavelengths_angstrom)}")
else:
    print("No spectroscopy data found for any star.")

Star ID: 1SWASP J164159.14+363818.5
Wavelength points: 562
